# 04 · Spaced repetition

Your 1–5 grade does two things. It averages into **Confidence** (column L, notebook 03),
and it moves the word's **FSRS schedule** (columns M–R), which decides *when* the word
comes back. FSRS is the algorithm Anki ships; the maths comes from the `fsrs` package, and
`flashcards/srs.py` maps it onto the workbook and onto the five buttons.

Covers `srs.review`, the "due" deck (`session.build_deck(mode="due")`) and the rule that
keeps in-session repeats from inflating a schedule.

In [1]:
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "experiments")]

import pandas as pd
from demo_workbook import make_demo_workbook
from flashcards import srs
from flashcards.workbook import load_cards

WORKBOOK = make_demo_workbook(ROOT / "experiments" / "_demo" / "demo_vocab.xlsx")
data = load_cards(str(WORKBOOK))
NOW = datetime(2026, 10, 2, 9, 0, tzinfo=timezone.utc)
UTC = timezone.utc
# Unfuzzed: real grades get a few percent of random spread so words learned
# together don't all fall due on the same day; here we want exact numbers.
sched = srs.make_scheduler(fuzz=False)

## 1. Five buttons, four FSRS ratings

FSRS knows Again / Hard / Good / Easy. The app keeps its five buttons: 3 *getting there*
and 4 *solid* are both a successful recall, so both are **Good**.

In [2]:
def first_review(grade):
    s = srs.review(None, grade, NOW, sched)
    return {"you press": grade, "FSRS rating": srs.GRADE_TO_RATING[grade].name,
            "state after": s.state, "next review": srs.describe_due(s.due, NOW, UTC)}

pd.DataFrame([first_review(g) for g in range(1, 6)])

,you press,FSRS rating,state after,next review
0,1,Again,Learning,in 1 min
1,2,Hard,Learning,in 6 min
2,3,Good,Learning,in 10 min
3,4,Good,Learning,in 10 min
4,5,Easy,Review,in 8 days


A new word graded 3–4 goes through two short **learning steps** (1 min, 10 min) before its
first real interval; *mastered* skips them.

## 2. A word over two months

Each review happens when the word is due. Good answers stretch the gap; one *Again*
resets it to a short relearning step, but FSRS keeps part of the history (stability): the
first interval after relearning is 5 days, not the 2 days a fresh word gets.

In [3]:
def history(grades):
    s, t, rows = None, NOW, []
    for g in grades:
        s = srs.review(s, g, t, sched)
        rows.append({"reviewed": t.strftime("%b %d %H:%M"), "grade": g, "state": s.state,
                     "stability (days)": round(s.stability, 1),
                     "next review": srs.describe_due(s.due, t, UTC)})
        t = s.due
    return pd.DataFrame(rows)

history([4, 4, 4, 4, 4, 1, 4, 4, 4])

,reviewed,grade,state,stability (days),next review
0,Oct 02 09:00,4,Learning,2.3,in 10 min
1,Oct 02 09:10,4,Review,2.3,in 2 days
2,Oct 04 09:10,4,Review,11.0,in 11 days
3,Oct 15 09:10,4,Review,46.3,in 46 days
4,Nov 30 09:10,4,Review,163.0,in 5 months
5,May 12 09:10,1,Relearning,4.7,in 10 min
6,May 12 09:20,4,Review,4.7,in 5 days
7,May 17 09:20,4,Review,12.1,in 12 days
8,May 29 09:20,4,Review,27.7,in 28 days


## 3. Repeats inside one session do not count twice

With *keep repeating* on, a word can come round several times in one sitting. Those
repeats move **Confidence**, but FSRS only takes the first grade of the session, plus any
grade given when FSRS itself has the word due again (a learning step after *Again*).
Without that rule, four same-day *Easy* presses pushed a new word a month out.

In [4]:
from flashcards.session import ReviewSession

word = next(c for c in data.cards if c.schedule is None)
naive = None
for _ in range(4):
    naive = srs.review(naive, 5, NOW, sched)        # every repeat fed to FSRS

s = ReviewSession([word], repeat=True, scheduler=sched)
while not s.finished:
    s.grade(5, NOW)
pd.DataFrame([
    {"rule": "every repeat counts (rejected)", "next review": srs.describe_due(naive.due, NOW, UTC)},
    {"rule": "first grade per session (in use)", "next review": srs.describe_due(s.schedules[word.id].due, NOW, UTC)},
    {"rule": "presses in the session", "next review": len(s.log)},
])

,rule,next review
0,every repeat counts (rejected),in 30 days
1,first grade per session (in use),in 8 days
2,presses in the session,3


## 4. The "Due now" deck

Due words first, the ones you are most likely to have forgotten (lowest estimated recall)
at the top, then words never scheduled. Words due later are left out. A word in the Review
state counts as due for its whole local day, as in Anki, so studying at 08:00 picks up a
word due at 21:00.

In [5]:
from dataclasses import replace
from flashcards.session import build_deck

# Pretend the first six words were learned at different times in the past.
past = [srs.review(None, 4, NOW - timedelta(days=d), sched) for d in (1, 3, 9, 20, 40, 2)]
for i, s in enumerate(list(past)):
    past[i] = srs.review(s, 4, s.due, sched)      # second review, when it was due
cards = [replace(c, schedule=past[i]) if i < len(past) else c for i, c in enumerate(data.cards)]

deck = build_deck(cards, mode="due", now=NOW, limit=10, seed=0, local_tz=UTC)
pd.DataFrame([{
    "word": c.term,
    "status": srs.status(c.schedule, NOW, UTC),
    "recall now": None if c.schedule is None else round(srs.retrievability(c.schedule, NOW, sched), 2),
} for c in deck])

,word,status,recall now
0,Unterlagen,due,0.64
1,Besprechung,due,0.71
2,Termin,due,0.80
3,Bericht,due,0.91
4,Schreibtisch,due,0.95
5,knapp,new,NaN
6,Umstieg,new,NaN
7,Unterkunft,new,NaN
8,Gelassenheit,new,NaN
9,Verspaetung,new,NaN


## 5. Where it is stored

Columns M–R of the word's own row, written by the same surgical write-back as Confidence
(notebook 03). Times are UTC text so Excel never reinterprets them; the headers are added
the first time a sheet gets a schedule.

In [6]:
import openpyxl
from flashcards.writeback import apply_updates

s = ReviewSession(deck[:3], scheduler=sched)
for g in (5, 3, 1):
    s.grade(g, NOW)
updates = s.cell_updates()
for sheet in {k[0] for k in updates} & set(data.sheets_missing_srs_headers):
    updates[(sheet, 1)] = dict(srs.SRS_HEADERS)
apply_updates(str(WORKBOOK), updates, original_mtime=data.mtime,
              backup_dir=str(ROOT / "experiments" / "_demo" / "backups"))

wb = openpyxl.load_workbook(WORKBOOK)
rows = []
for c in deck[:3]:
    ws = wb[c.sheet]
    rows.append({"word": c.term, **{ws[f"{col}1"].value: ws[f"{col}{c.row}"].value
                                    for col in ("L", *srs.SRS_COLUMNS)}})
pd.DataFrame(rows)

,word,Confidence (1-5),Due (FSRS),State (FSRS),Step (FSRS),Stability (FSRS),Difficulty (FSRS),Last review (FSRS)
0,Unterlagen,3.5,2026-12-15T09:00:00Z,Review,NaN,73.7640,1.0000,2026-10-02T09:00:00Z
1,Besprechung,3.0,2026-11-03T09:00:00Z,Review,NaN,32.2265,2.1043,2026-10-02T09:00:00Z
2,Termin,1.0,2026-10-02T09:10:00Z,Relearning,0.0,0.7330,7.3922,2026-10-02T09:00:00Z


Back to [01 · Reading the workbook](01_reading_the_workbook.ipynb) ·
[02 · Choosing what to study](02_choosing_what_to_study.ipynb) ·
[03 · Grading and writing back](03_grading_and_writeback.ipynb)